# 04 · `conditional_edges` + node routing

Intent router that classifies the user's question and sends it to the correct specialized node.

```
"Explain what a vector store is"    → knowledge  → LLM responds directly
"Search docs about LangChain"       → search     → search tool
"Summarize this text: ..."          → summarize  → summary tool
```

The router **does not use an LLM** — classification is deterministic (pure conditional).
This pattern is the core of any multi-agent system (Supervisor).

In [1]:
from dotenv import load_dotenv

load_dotenv("../.env")

True

In [ ]:
from typing import Annotated, TypedDict

from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.graph import END, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

## State + Tools

In [ ]:
class State(TypedDict):
    messages: Annotated[list, add_messages]


@tool
def search_docs(query: str) -> str:
    """Searches documentation for a given query."""
    return f"Found 3 docs about '{query}': intro, api-ref, tutorial."


@tool
def summarize_text(text: str) -> str:
    """Summarizes the given text."""
    return f"Summary: {text[:80]}..."


llm = ChatOpenAI(model="gpt-4o-mini")

## Deterministic router + specialized nodes

In [ ]:
def classify(state: State) -> str:
    """Deterministic router — classifies by keywords, no LLM."""
    last = state["messages"][-1].content.lower()
    if "search" in last:
        return "search_agent"
    if "summarize" in last:
        return "summarize_agent"
    return "knowledge_agent"


def router(state: State):
    """Router node — just passes the state through; logic is in classify."""
    return state


def knowledge_agent(state: State):
    """Responds directly with the LLM — no tools."""
    response = llm.invoke(state["messages"])
    return {"messages": [response]}


def search_agent(state: State):
    """LLM with search tool."""
    llm_with_search = llm.bind_tools([search_docs])
    response = llm_with_search.invoke(state["messages"])
    return {"messages": [response]}


def summarize_agent(state: State):
    """LLM with summarize tool."""
    llm_with_summarize = llm.bind_tools([summarize_text])
    response = llm_with_summarize.invoke(state["messages"])
    return {"messages": [response]}

## Build the graph with 3 paths

In [ ]:
builder = StateGraph(State)

# Nodos
builder.add_node("router", router)
builder.add_node("knowledge_agent", knowledge_agent)
builder.add_node("search_agent", search_agent)
builder.add_node("summarize_agent", summarize_agent)
builder.add_node("search_tools", ToolNode([search_docs]))
builder.add_node("summarize_tools", ToolNode([summarize_text]))

# Entry point
builder.set_entry_point("router")

# Router → 3 caminos
builder.add_conditional_edges("router", classify, {
    "search_agent": "search_agent",
    "summarize_agent": "summarize_agent",
    "knowledge_agent": "knowledge_agent",
})

# knowledge responde directo → END
builder.add_edge("knowledge_agent", END)

# search puede llamar tools
def search_should_continue(state: State):
    if state["messages"][-1].tool_calls:
        return "search_tools"
    return END

builder.add_conditional_edges("search_agent", search_should_continue)
builder.add_edge("search_tools", "search_agent")

# summarize puede llamar tools
def summarize_should_continue(state: State):
    if state["messages"][-1].tool_calls:
        return "summarize_tools"
    return END

builder.add_conditional_edges("summarize_agent", summarize_should_continue)
builder.add_edge("summarize_tools", "summarize_agent")

graph = builder.compile()

## Visualize the graph (3 paths)

In [6]:
graph.get_graph().print_ascii()

                              +-----------+                                
                              | __start__ |                                
                              +-----------+                                
                                     *                                     
                                     *                                     
                                     *                                     
                                +--------+                                 
                              ..| router |...                              
                          ....  +--------+   ....                          
                     .....           .           .....                     
                 ....                .                ....                 
              ...                    .                    ...              
+-----------------+          +--------------+          +-----------------+ 
| knowledge_

## Test all 3 paths

In [ ]:
# Path 1: knowledge
r1 = graph.invoke({"messages": [{"role": "user", "content": "Explain what a vector store is"}]})
print("KNOWLEDGE:", r1["messages"][-1].content[:120])
print("---")

In [ ]:
# Path 2: search
r2 = graph.invoke({"messages": [{"role": "user", "content": "Search docs about LangChain"}]})
for msg in r2["messages"]:
    print(f"{msg.__class__.__name__}: {msg.content[:120]}")
print("---")

In [ ]:
# Path 3: summarize
r3 = graph.invoke({"messages": [{"role": "user", "content": "Summarize this text: LangGraph is a framework for building agents with state graphs."}]})
for msg in r3["messages"]:
    print(f"{msg.__class__.__name__}: {msg.content[:120]}")
print("---")